# Centroides de crimen por código postal y tipo de crimen

Notebook autocontenido para generar un CSV de centroides con K-Means por **código postal (CP)** y **tipo de crimen**.

- Lee el CSV de carpetas FGJ CDMX con columnas como `delito`, `categoria_delito`, `longitud`, `latitud`.
- Asigna CP con un GeoJSON de códigos postales cuando el CSV no trae CP.
- Calcula centroides por grupo `(cp, tipo_crimen)` usando K-Means, priorizando pocos centroides.
- Exporta un CSV con coordenadas del centroide, conteos y radios de dispersión.

> Ajusta únicamente la celda de configuración y ejecuta todo.



In [ ]:
# Si estás en Colab o un ambiente limpio, descomenta esta línea:
# %pip install -q pandas geopandas shapely pyogrio rtree scikit-learn pyproj


In [ ]:
from __future__ import annotations

import re
import unicodedata
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd
from pyproj import Transformer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# =============================================================================
# CONFIGURACIÓN: cambia estas rutas/columnas según tu notebook
# =============================================================================
INPUT_FILE = Path("/content/da_carpetas-de-investigacion-pgj-cdmx.csv")
CP_GEOJSON_FILE = Path("/content/geojson_cp/open_mexico_sepomeX_cdmx.geojson")
OUTPUT_CSV = Path("/content/city_zen_outputs/centroides_crimen_cp.csv")

LAT_COL = "latitud"
LON_COL = "longitud"
CP_GEOJSON_COL = "d_codigo"

# Si tu CSV ya trae código postal, pon aquí el nombre de columna, por ejemplo "cp".
# Si es None, el notebook asigna CP por cruce espacial con el GeoJSON.
EXISTING_CP_COL: str | None = None

# Tipo de crimen para agrupar:
# - None: crea una categoría refinada estable a partir de categoria_delito + delito.
# - "delito": genera centroides por delito crudo.
# - "categoria_delito": genera centroides por categoría cruda.
CRIME_TYPE_COL: str | None = None

# Controles para priorizar pocos centroides por cada grupo (CP, tipo_crimen).
MAX_CENTROIDS_PER_GROUP = 3
MIN_EVENTS_TO_SPLIT = 25
MIN_EVENTS_PER_CENTROID = 15
MIN_SILHOUETTE_TO_SPLIT = 0.20
SILHOUETTE_IMPROVEMENT = 0.04
RANDOM_STATE = 42

# Bounding box aproximado de CDMX; filtra coordenadas nulas/0 o claramente fuera de rango.
CDMX_BBOX = {
    "lat_min": 19.00,
    "lat_max": 19.75,
    "lon_min": -99.45,
    "lon_max": -98.80,
}

# Proyección métrica para clustering en metros. EPSG:32614 = UTM zona 14N, adecuada para CDMX.
WGS84_CRS = "EPSG:4326"
METRIC_CRS = "EPSG:32614"

INCIDENT_RULES: tuple[tuple[str, str], ...] = (
    ("Homicidio", r"\b(homicidio|feminicidio)\b"),
    ("Lesiones", r"\b(lesion|lesiones)\b"),
    ("Robo a transeúnte", r"\b(robo).*\b(transeunte|peaton|via publica)\b"),
    ("Robo de vehículo", r"\b(robo).*\b(vehiculo|automovil|motocicleta|auto|moto)\b"),
    ("Robo a negocio", r"\b(robo).*\b(negocio|comercio|tienda|establecimiento)\b"),
    ("Robo a casa habitación", r"\b(robo).*\b(casa habitacion|domicilio|vivienda)\b"),
    ("Robo en transporte", r"\b(robo).*\b(transporte|metro|microbus|taxi|combi|autobus)\b"),
    ("Violencia familiar", r"\b(violencia familiar|violencia intrafamiliar)\b"),
    ("Delitos sexuales", r"\b(violacion|abuso sexual|acoso sexual|hostigamiento sexual)\b"),
    ("Narcomenudeo", r"\b(narcomenudeo|posesion de droga|contra la salud)\b"),
    ("Fraude y extorsión", r"\b(fraude|extorsion)\b"),
    ("Daño a la propiedad", r"\b(dano|danio|danos|danios).*\b(propiedad|bien|bienes)\b"),
    ("Amenazas", r"\b(amenaza|amenazas)\b"),
    ("Secuestro", r"\b(secuestro|privacion de la libertad)\b"),
)


def strip_accents(value: object) -> str:
    """Normaliza texto para comparaciones robustas ante acentos y mayúsculas."""
    text = "" if pd.isna(value) else str(value)
    text = unicodedata.normalize("NFKD", text)
    text = "".join(ch for ch in text if not unicodedata.combining(ch))
    return re.sub(r"\s+", " ", text.lower()).strip()


def normalize_cp(series: pd.Series) -> pd.Series:
    """Extrae CP de 5 dígitos conservando ceros a la izquierda cuando existan."""
    return series.astype(str).str.extract(r"(\d{5})", expand=False).astype("string")


def refine_incident_type(row: pd.Series) -> str:
    """Agrupa `categoria_delito` y `delito` en categorías estables para no explotar tipos crudos."""
    raw_text = " ".join(
        strip_accents(row[col])
        for col in ("categoria_delito", "delito")
        if col in row.index and pd.notna(row[col])
    )
    if not raw_text:
        return "Sin tipificar"
    for label, pattern in INCIDENT_RULES:
        if re.search(pattern, raw_text):
            return label
    if "robo" in raw_text:
        return "Otros robos"
    return "Otros incidentes"


def load_cp_geojson(path: Path) -> gpd.GeoDataFrame:
    """Carga el GeoJSON de códigos postales y estandariza la columna `cp`."""
    gdf = gpd.read_file(path)
    if CP_GEOJSON_COL not in gdf.columns:
        raise ValueError(f"El GeoJSON no contiene la columna {CP_GEOJSON_COL!r}. Columnas: {list(gdf.columns)}")
    gdf = gdf.to_crs(WGS84_CRS).copy()
    gdf["cp"] = normalize_cp(gdf[CP_GEOJSON_COL])
    gdf = gdf[gdf["cp"].notna() & ~gdf.geometry.is_empty & gdf.geometry.notna()].copy()
    return gdf[["cp", "geometry"]]


def prepare_crime_points(df: pd.DataFrame) -> pd.DataFrame:
    """Limpia coordenadas y crea/normaliza `cp` y `crime_type`."""
    df = df.copy()
    for col in (LAT_COL, LON_COL):
        if col not in df.columns:
            raise ValueError(f"No encontré la columna requerida {col!r}. Columnas: {list(df.columns)}")
        df[col] = pd.to_numeric(df[col], errors="coerce")

    valid_coords = (
        df[LAT_COL].between(CDMX_BBOX["lat_min"], CDMX_BBOX["lat_max"])
        & df[LON_COL].between(CDMX_BBOX["lon_min"], CDMX_BBOX["lon_max"])
        & ~((df[LAT_COL] == 0) & (df[LON_COL] == 0))
    )
    df = df[valid_coords].copy()

    if EXISTING_CP_COL and EXISTING_CP_COL in df.columns:
        df["cp"] = normalize_cp(df[EXISTING_CP_COL])
    else:
        df["cp"] = pd.Series(pd.NA, index=df.index, dtype="string")

    if CRIME_TYPE_COL:
        if CRIME_TYPE_COL not in df.columns:
            raise ValueError(f"CRIME_TYPE_COL={CRIME_TYPE_COL!r} no existe. Columnas: {list(df.columns)}")
        df["crime_type"] = df[CRIME_TYPE_COL].fillna("Sin tipificar").astype(str).str.strip().replace("", "Sin tipificar")
    else:
        df["crime_type"] = df.apply(refine_incident_type, axis=1)

    return df


def assign_cp_spatial(df: pd.DataFrame, cp_gdf: gpd.GeoDataFrame) -> pd.DataFrame:
    """Asigna CP por punto-en-polígono solo a registros que no tienen CP válido."""
    df = df.copy()
    needs_cp = df["cp"].isna()
    if not needs_cp.any():
        df["cp_match_method"] = "existing_cp"
        return df

    gdf_points = gpd.GeoDataFrame(
        df.loc[needs_cp].copy(),
        geometry=gpd.points_from_xy(df.loc[needs_cp, LON_COL], df.loc[needs_cp, LAT_COL]),
        crs=WGS84_CRS,
    )
    joined = gpd.sjoin(gdf_points, cp_gdf, how="left", predicate="within", rsuffix="poly")
    joined = joined[~joined.index.duplicated(keep="first")]
    cp_join_col = "cp_poly" if "cp_poly" in joined.columns else "cp"

    df["cp_match_method"] = np.where(df["cp"].notna(), "existing_cp", "missing_cp")
    has_spatial_cp = joined[cp_join_col].notna()
    df.loc[joined.index[has_spatial_cp], "cp"] = joined.loc[has_spatial_cp, cp_join_col].astype("string")
    df.loc[joined.index[has_spatial_cp], "cp_match_method"] = "spatial_within"
    return df


def choose_k(points_xy: np.ndarray) -> tuple[int, str]:
    """Elige un K pequeño: por default 1; sube a 2-3 solo si hay suficientes datos y mejora clara."""
    n = len(points_xy)
    if n < MIN_EVENTS_TO_SPLIT:
        return 1, f"k=1 porque n={n} < MIN_EVENTS_TO_SPLIT={MIN_EVENTS_TO_SPLIT}"

    max_allowed = min(MAX_CENTROIDS_PER_GROUP, n // MIN_EVENTS_PER_CENTROID)
    if max_allowed < 2:
        return 1, f"k=1 porque n={n} no alcanza MIN_EVENTS_PER_CENTROID={MIN_EVENTS_PER_CENTROID}"

    best_k = 1
    best_score = -1.0
    best_reason = "k=1 base; ningún split superó los umbrales de silueta"
    previous_score = None

    for k in range(2, max_allowed + 1):
        labels = KMeans(n_clusters=k, n_init="auto", random_state=RANDOM_STATE).fit_predict(points_xy)
        if len(set(labels)) < 2:
            continue
        score = silhouette_score(points_xy, labels)
        improvement = score if previous_score is None else score - previous_score
        passes_threshold = score >= MIN_SILHOUETTE_TO_SPLIT and (
            previous_score is None or improvement >= SILHOUETTE_IMPROVEMENT
        )
        if score > best_score and passes_threshold:
            best_k = k
            best_score = score
            best_reason = f"k={k}; silhouette={score:.3f}; mejora={improvement:.3f}"
        previous_score = score

    return best_k, best_reason


def cluster_one_group(group: pd.DataFrame, transformer_to_wgs84: Transformer) -> list[dict]:
    """Calcula centroides para un grupo CP + tipo de crimen."""
    points_xy = group[["x_m", "y_m"]].to_numpy()
    k, reason = choose_k(points_xy)
    model = KMeans(n_clusters=k, n_init="auto", random_state=RANDOM_STATE)
    labels = model.fit_predict(points_xy) if k > 1 else np.zeros(len(group), dtype=int)
    centers_xy = model.cluster_centers_ if k > 1 else points_xy.mean(axis=0, keepdims=True)

    rows: list[dict] = []
    total_group = len(group)
    for cluster_id in range(k):
        mask = labels == cluster_id
        cluster_points = points_xy[mask]
        center_x, center_y = centers_xy[cluster_id]
        centroid_lon, centroid_lat = transformer_to_wgs84.transform(center_x, center_y)
        distances_m = np.sqrt(((cluster_points - np.array([center_x, center_y])) ** 2).sum(axis=1))
        rows.append(
            {
                "cp": group["cp"].iloc[0],
                "crime_type": group["crime_type"].iloc[0],
                "centroid_id": cluster_id + 1,
                "centroids_in_group": k,
                "centroid_lon": round(float(centroid_lon), 7),
                "centroid_lat": round(float(centroid_lat), 7),
                "n_events": int(mask.sum()),
                "pct_events_group": round(float(mask.sum() / total_group * 100), 2),
                "radius_p50_m": round(float(np.percentile(distances_m, 50)), 1),
                "radius_p90_m": round(float(np.percentile(distances_m, 90)), 1),
                "kmeans_reason": reason,
            }
        )
    return rows


def build_centroids() -> pd.DataFrame:
    """Pipeline completo: lee datos, asigna CP, clusteriza y guarda OUTPUT_CSV."""
    OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)

    crimes = pd.read_csv(INPUT_FILE, low_memory=False)
    cp_gdf = load_cp_geojson(CP_GEOJSON_FILE)
    crimes = prepare_crime_points(crimes)
    crimes = assign_cp_spatial(crimes, cp_gdf)
    crimes = crimes[crimes["cp"].notna()].copy()

    if crimes.empty:
        raise ValueError("No quedaron registros con coordenadas válidas y CP asignado.")

    points_gdf = gpd.GeoDataFrame(
        crimes,
        geometry=gpd.points_from_xy(crimes[LON_COL], crimes[LAT_COL]),
        crs=WGS84_CRS,
    ).to_crs(METRIC_CRS)
    points_gdf["x_m"] = points_gdf.geometry.x
    points_gdf["y_m"] = points_gdf.geometry.y

    transformer_to_wgs84 = Transformer.from_crs(METRIC_CRS, WGS84_CRS, always_xy=True)
    rows: list[dict] = []
    for (_, _), group in points_gdf.groupby(["cp", "crime_type"], sort=True):
        rows.extend(cluster_one_group(group, transformer_to_wgs84))

    centroids = pd.DataFrame(rows).sort_values(["cp", "crime_type", "centroid_id"]).reset_index(drop=True)
    centroids.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

    print(f"Registros usados: {len(points_gdf):,}")
    print(f"Grupos CP + tipo_crimen: {points_gdf.groupby(['cp', 'crime_type']).ngroups:,}")
    print(f"Centroides generados: {len(centroids):,}")
    print(f"CSV escrito en: {OUTPUT_CSV}")
    return centroids


centroids_df = build_centroids()
centroids_df.head(20)




In [ ]:
# Revisión rápida de cuántos centroides se generaron por grupo.
summary = (
    centroids_df.groupby("centroids_in_group")
    .size()
    .rename("num_centroides")
    .reset_index()
    .sort_values("centroids_in_group")
)
summary


In [ ]:
# En Google Colab, descarga el CSV generado.
try:
    from google.colab import files

    files.download(str(OUTPUT_CSV))
except ImportError:
    print(f"Archivo listo en: {OUTPUT_CSV}")
